In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from autogluon.tabular import TabularDataset, TabularPredictor

In [6]:
df = pd.read_csv("../data/creditcard.csv")

In [10]:
!python -m pip freeze

absl-py==2.5.0
accelerate==1.15.0
adagio==0.2.6
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiohttp-cors==0.8.1
aiosignal==1.4.0
alembic==1.20.0
annotated-doc==0.0.5
annotated-types==0.8.0
antlr4-python3-runtime==4.9.3
anyio==4.15.1
appnope==1.0.0
apsw==3.53.4.0
apswutils==0.1.2
asttokens==3.0.2
attrs==26.1.0
autogluon==1.6.3
autogluon.common==1.6.3
autogluon.core==1.6.3
autogluon.features==1.6.3
autogluon.multimodal==1.6.3
autogluon.tabular==1.6.3
autogluon.timeseries==1.6.3
beartype==0.22.9
beautifulsoup4==4.15.0
blis==1.3.3
boto3==1.43.105
botocore==1.43.105
catalogue==2.0.10
catboost==1.2.10
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
chronos-forecasting==2.3.2
click==8.5.0
cloudpathlib==0.25.0
cloudpickle==3.1.2
colorama==0.4.6
colorful==0.5.8
colorlog==6.12.0
comm==0.2.3
confection==1.3.3
contourpy==1.4.0
coreforecast==0.0.16
cryptography==50.0.1
cycler==0.12.1
cymem==2.0.13
datasets==5.0.1
debugpy==1.8.22
defusedxml==0.7.1
dill==0.4.1
distlib==0.4.3
einops==0.8.2
einx==

In [ ]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# INSTALAR AUTOGLUON
# =========================
#%pip install autogluon

for SEED in [42, 101, 223, 302, 404]:
    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
)

    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    train_data = TabularDataset(train_df)
    test_data = TabularDataset(test_df)

    # =========================
    # AUTOGLUON
    # =========================
    predictor = TabularPredictor(
        label='Class'
    ).fit(
        train_data=train_data
    )

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    leaderboard = predictor.leaderboard(test_data, display=True)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    resultados = []

    X_test_ag = test_df.drop(columns=['Class'])
    y_true = test_df['Class'].values

    # Iterar sobre todos os modelos gerados (incluindo Ensembles)
    for model_name in predictor.model_names():

        # Previsão específica para cada modelo
        preds = predictor.predict(X_test_ag, model=model_name)
        y_pred = preds.astype(int).values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        average_precision = round(average_precision_score(y_true, y_pred, pos_label=1, average='weighted'), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name.split('_')[0],
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted,
            "average_precision": average_precision
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS (SEED: {})=====".format(seed))
    print(df_resultados.sort_values(by="average_precision", ascending=False))

No path specified. Models will be saved in: "AutogluonModels/ag-20261029_213743"


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.05 GB / 16.00 GB (19.0%)
Disk Space Avail:   190.21 GB / 460.38 GB (41.3%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/Inatel/TCC-autoML/notebooks/AutogluonModels/ag-20261029_213743"
Train Data Rows:    227845
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your prediction problem is: 'binary' (becaus


===== LEADERBOARD =====
                  model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        ExtraTreesGini    0.891304   0.857143          f1        0.126301       0.039375   3.805021                 0.126301                0.039375           3.805021            1       True          6
1      RandomForestEntr    0.890110   0.666667          f1        0.070841       0.051125  30.496402                 0.070841                0.051125          30.496402            1       True          4
2      RandomForestGini    0.885246   0.666667          f1        0.092261       0.026262  44.329668                 0.092261                0.026262          44.329668            1       True          3
3        ExtraTreesEntr    0.885246   0.857143          f1        0.113179       0.040608   3.540779                 0.113179                0.040608          